# ICL-PILOT Direct Generation Colab

This notebook runs the repo's native generation pipeline directly with `transformers`.

It keeps the existing experimental design fixed:
- age-matched SLI-TD frozen rosters per cohort
- the same counterbalanced story schedule
- the same canonical ENNI story spines
- the same two-stage workflow: stage 1 TD-like story, then stage 2 SLI-like transformation

This version builds and runs **all requested cohorts in one pass** (default: 5- to 9-year-olds).


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/shamira-venturini/ICL-PILOT.git"
REPO_DIR = Path("/content/ICL-PILOT")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/ICL-PILOT

In [ ]:
%pip -q install -U pip setuptools wheel
%pip -q install -e . transformers accelerate bitsandbytes huggingface_hub sentencepiece

In [ ]:
import sys
from pathlib import Path

SRC_DIR = Path('/content/ICL-PILOT/src')
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import icl_pilot
print('icl_pilot import OK from', icl_pilot.__file__)

## Model And Run Settings

The report names `Meta-Llama-3.2-8B-Instruct` as the generation model. On Hugging Face, the usual repo id is `meta-llama/Llama-3.2-8B-Instruct`. If the model is gated in your environment, set `HF_TOKEN` first.

In [ ]:
import os
from getpass import getpass

# Uncomment if needed for gated model access.
# os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")

MODEL_NAME = "avans06/Meta-Llama-3.2-8B-Instruct"
SEED = 42
TEMPERATURE = 0.7
DO_SAMPLE = True
MAX_NEW_TOKENS_STAGE1 = 180
MAX_NEW_TOKENS_STAGE2 = 180
N_REPLICATES = 10
PAIRS_PER_ROUND = 3
AGE_BINS = [
    ("5-year-old", 60, 71),
    ("6-year-old", 72, 83),
    ("7-year-old", 84, 95),
    ("8-year-old", 96, 107),
    ("9-year-old", 108, 119),
]
ROW_LIMIT = None
SAVE_EVERY = 25

print({
    "MODEL_NAME": MODEL_NAME,
    "SEED": SEED,
    "TEMPERATURE": TEMPERATURE,
    "N_REPLICATES": N_REPLICATES,
    "PAIRS_PER_ROUND": PAIRS_PER_ROUND,
    "AGE_BINS": AGE_BINS,
    "ROW_LIMIT": ROW_LIMIT,
})


In [ ]:
import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print({
    "cuda_available": torch.cuda.is_available(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
})

In [ ]:
from pathlib import Path
from icl_pilot.generation import build_multi_age_rotating_generation_packages


def first_existing(paths):
    for path in paths:
        if Path(path).exists():
            return path
    raise FileNotFoundError(f"None of these paths exists: {paths}")

DEV_MEASURES_CSV = first_existing([
    "phase2/ENNI/measures/dev_measures.csv",
    "phase2/measures/dev_measures.csv",
])
SEVERITY_PROFILE_CSV = first_existing([
    "phase2/ENNI/measures/severity_profile_banded_table.csv",
    "phase2/measures/severity_profile_banded_table.csv",
])
COUNTERBALANCE_CSV = "configs/generation/cross_story_counterbalancing.csv"

multi_age_artifacts = build_multi_age_rotating_generation_packages(
    dev_measures_csv=DEV_MEASURES_CSV,
    severity_profile_csv=SEVERITY_PROFILE_CSV,
    counterbalance_csv=COUNTERBALANCE_CSV,
    output_dir="configs/generation/colab_generation_package_direct_rotating_multi_age",
    age_bins=AGE_BINS,
    n_replicates=N_REPLICATES,
    pairs_per_round=PAIRS_PER_ROUND,
)

print(f"Summary: {multi_age_artifacts.summary_json}")
for cohort_label, artifacts in multi_age_artifacts.cohort_packages.items():
    print(cohort_label, "->", artifacts.schedule_csv)

multi_age_artifacts


In [ ]:
import json
import pandas as pd
from pathlib import Path

all_schedule = []
all_eval_manifest = []
for cohort_label, artifacts in multi_age_artifacts.cohort_packages.items():
    cohort_schedule = pd.read_csv(artifacts.schedule_csv)
    cohort_eval = pd.read_csv(artifacts.eval_manifest_csv)
    cohort_schedule["cohort_label"] = cohort_label
    cohort_eval["cohort_label"] = cohort_label
    all_schedule.append(cohort_schedule)
    all_eval_manifest.append(cohort_eval)

schedule_df = pd.concat(all_schedule, ignore_index=True)
eval_manifest_df = pd.concat(all_eval_manifest, ignore_index=True)


def first_existing(paths):
    for path in paths:
        if Path(path).exists():
            return path
    raise FileNotFoundError(f"None of these paths exists: {paths}")

narratives_csv = first_existing([
    "phase2/ENNI/measures/story_units/story_unit_narratives.csv",
    "phase2/story_units/story_unit_narratives.csv",
])
story_reference_json = first_existing([
    "phase2/ENNI/measures/story_units/story_unit_reference.json",
    "phase2/story_units/story_unit_reference.json",
])

narratives_df = pd.read_csv(narratives_csv)
story_reference = json.loads(Path(story_reference_json).read_text())

def numbered_lines(lines):
    return "\n".join(f"{i + 1}. {line}" for i, line in enumerate(lines))

def bullet_lines(lines):
    if not lines:
        return "None"
    return "\n".join(f"- {line}" for line in lines)

def exemplar_lookup(frame):
    table = frame[["File_ID", "Group", "story_id", "story_text"]].copy()
    table["File_ID"] = table["File_ID"].astype(str)
    table["Group"] = table["Group"].astype(str)
    table["story_id"] = table["story_id"].astype(str)
    return {
        (row.File_ID, row.Group, row.story_id): row.story_text
        for row in table.itertuples(index=False)
    }

story_lookup = exemplar_lookup(narratives_df)

run_df = schedule_df.copy()
run_df["story_title"] = run_df["target_story"].map(lambda story_id: story_reference[story_id]["title"])
run_df["story_units_text"] = run_df["target_story"].map(
    lambda story_id: numbered_lines(story_reference[story_id]["units"])
)
run_df["content_invariants_text"] = run_df["target_story"].map(
    lambda story_id: bullet_lines(story_reference[story_id].get("content_invariants", []))
)
run_df["optional_content_text"] = run_df["target_story"].map(
    lambda story_id: bullet_lines(story_reference[story_id].get("optional_content", []))
)
run_df["prompt_constraints_text"] = run_df["target_story"].map(
    lambda story_id: bullet_lines(story_reference[story_id].get("prompt_constraints", []))
)
run_df["prompt_td_exemplar_story"] = run_df.apply(
    lambda row: story_lookup.get((str(int(row["td_child_id"])), "TD", str(row["prompt_td_story_e1"])), ""),
    axis=1,
)
run_df["prompt_sli_exemplar_story"] = run_df.apply(
    lambda row: story_lookup.get((str(int(row["sli_child_id"])), "SLI", str(row["prompt_sli_story_e2"])), ""),
    axis=1,
)

if ROW_LIMIT is not None:
    run_df = run_df.head(ROW_LIMIT).copy()

print(f"Rows to generate: {len(run_df)}")
display(run_df[[
    "cohort_label",
    "round_id",
    "bundle_id",
    "replicate_id",
    "target_story",
    "story_title",
    "sli_child_id",
    "td_child_id",
    "prompt_td_story_e1",
    "prompt_sli_story_e2",
]].head())

display(eval_manifest_df[[
    "cohort_label",
    "round_id",
    "eval_group",
    "comparison_target_group",
    "eval_pair_id",
    "eval_child_id",
    "bundle_id",
]].head(12))


## Load The Model

The default path tries 4-bit loading on GPU so the 8B model is more likely to fit on Colab. If you have a larger GPU, you can simplify this cell.

In [ ]:
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_4bit = torch.cuda.is_available()
quant_config = None
model_kwargs = {
    "device_map": "auto",
}

if use_4bit:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    )
    model_kwargs["quantization_config"] = quant_config
else:
    model_kwargs["torch_dtype"] = torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.eval()

print({
    "loaded_model": MODEL_NAME,
    "use_4bit": use_4bit,
    "pad_token": tokenizer.pad_token,
})

In [ ]:
def stage1_messages(row):
    return [
        {
            "role": "system",
            "content": (
                "You write short child-like ENNI narratives. Preserve the picture-supported event order, "
                "allow small surface variation, and do not add new major plot events."
            ),
        },
        {
            "role": "user",
            "content": f"""Generate a child retell for the ENNI story below.

Target story: {row.target_story} ({row.story_title})
Target story slot in the schedule: {row.target_story}
Cross-story TD exemplar slot: {row.prompt_td_story_e1}
Replicate id: {row.replicate_id}

Canonical story spine:
{row.story_units_text}

Content that should stay stable:
{row.content_invariants_text}

Optional details:
{row.optional_content_text}

Constraints:
{row.prompt_constraints_text}

Example natural TD story from another ENNI story slot:
{row.prompt_td_story_e1}: {row.prompt_td_exemplar_story}

Write one short child-like narrative.
Keep the story events in order.
Do not add new major events.
Do not write an adult-style polished retell.
Use the example mainly for narrative style and level, not for its content.
Return only the narrative text.
""",
        },
    ]


def stage2_messages(row, stage1_text):
    return [
        {
            "role": "system",
            "content": (
                "You transform child narratives while preserving the underlying ENNI events and order. "
                "Make the output clinically plausible rather than caricatured."
            ),
        },
        {
            "role": "user",
            "content": f"""Rewrite the child story below so it sounds more like the example narrative while preserving the same ENNI events.

Target story: {row.target_story} ({row.story_title})
Cross-story exemplar slot: {row.prompt_sli_story_e2}

Example natural story from another ENNI story slot:
{row.prompt_sli_story_e2}: {row.prompt_sli_exemplar_story}

Canonical story spine:
{row.story_units_text}

Content that should stay stable:
{row.content_invariants_text}

Original child story:
{stage1_text}

Instructions:
- Preserve the same major events and event order.
- Keep the narration child-like.
- Let the example guide the style, grammatical level, and amount of detail.
- You may shorten clauses and omit optional details.
- Use the exemplar mainly as a style guide, not as content to copy.
- Avoid bizarre errors, random topic drift, or new plot events.
- Return only the transformed narrative text.
""",
        },
    ]


In [ ]:
def generate_from_messages(messages, max_new_tokens=180):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return text.strip()


In [ ]:
example_row = run_df.iloc[0]
stage1_preview_prompt = tokenizer.apply_chat_template(
    stage1_messages(example_row),
    tokenize=False,
    add_generation_prompt=True,
)
print(stage1_preview_prompt[:4000])

In [ ]:
from tqdm.auto import tqdm
from pathlib import Path
import re
import shutil
import zipfile
import errno
import subprocess

from icl_pilot.generated_chat import convert_generated_story_csv_to_cha

out_dir = Path("data/generated/direct_generation")
out_dir.mkdir(parents=True, exist_ok=True)

aggregate_path = out_dir / "five_to_nine_year_old_rotating_story_generations.csv"
if aggregate_path.exists():
    aggregate_path.unlink()

# Pipeline controls
DOWNLOAD_EACH_COHORT_CSV = False
CONVERT_TO_CHA = True
RUN_TALKTAG = False
# Match TalkTag_test default: annotate stage2_sli first.
# Set to ["stage1_td", "stage2_sli"] to run both.
TALKTAG_STAGES = ["stage2_sli"]
START_COHORT_LABEL = "6-year-old"
SAVE_TO_DRIVE = True
DOWNLOAD_CHA_AND_TALKTAG = True
DELETE_LOCAL_AFTER_EXPORT = True

# TalkTag CLI template (edit to match your TalkTag_test.ipynb command)
# Placeholders supported: {input_dir}, {output_dir}
TALKTAG_CMD_TEMPLATE = [
    "python", "-m", "talk_tag.cli", "annotate",
    "--input-dir", "{input_dir}",
    "--output-dir", "{output_dir}",
    "--target-speaker", "*CHI",
    "--speaker-field", "speaker",
    "--text-field", "utterance",
    "--limit", "0",
]

try:
    from google.colab import files as colab_files  # type: ignore
except Exception:
    colab_files = None


def cohort_slug(label: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", str(label).lower()).strip("_")
    return slug or "cohort"


def maybe_download(path: Path):
    if colab_files is None:
        print(f"Colab download unavailable; kept local file: {path}")
        return
    try:
        colab_files.download(str(path))
        print(f"Download started: {path.name}")
    except Exception as exc:
        print(f"Download failed for {path.name}: {exc}")


def resolve_roster_csv_for_cohort(label: str) -> str | None:
    try:
        package = multi_age_artifacts.cohort_packages.get(label)
        if package is None:
            return None
        roster = Path(package.roster_csv)
        return str(roster) if roster.exists() else None
    except Exception:
        return None


def run_talktag(input_dir: Path, output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    stage_map = {
        "stage1_td": (input_dir / "stage1_td", output_dir / "stage1_td"),
        "stage2_sli": (input_dir / "stage2_sli", output_dir / "stage2_sli"),
    }

    for stage_name in TALKTAG_STAGES:
        if stage_name not in stage_map:
            print(f"Skipping unknown TalkTag stage: {stage_name}")
            continue

        stage_input, stage_output = stage_map[stage_name]
        if not stage_input.exists():
            print(f"Skipping missing TalkTag input dir: {stage_input}")
            continue

        stage_output.mkdir(parents=True, exist_ok=True)
        cmd = [
            part.format(input_dir=str(stage_input), output_dir=str(stage_output))
            for part in TALKTAG_CMD_TEMPLATE
        ]
        print("TalkTag command:")
        print(" ".join(cmd))

        completed = subprocess.run(cmd, check=False, capture_output=True, text=True)

        log_path = output_dir / f"talktag_{stage_name}.log"
        log_text = (
            f"command: {' '.join(cmd)}\n"
            f"exit_code: {completed.returncode}\n\n"
            f"=== STDOUT ===\n{completed.stdout}\n"
            f"=== STDERR ===\n{completed.stderr}\n"
        )
        log_path.write_text(log_text, encoding="utf-8")
        print(f"TalkTag log: {log_path}")

        if completed.returncode != 0:
            tail = (completed.stderr or completed.stdout or "").strip().splitlines()[-12:]
            tail_text = "\n".join(tail)
            raise RuntimeError(
                f"TalkTag failed for {stage_name} with exit code {completed.returncode}. "
                f"See {log_path}.\n{tail_text}"
            )


def safe_zip_dir(src_dir: Path, zip_path: Path) -> Path | None:
    try:
        free_bytes = shutil.disk_usage(out_dir).free
        est_bytes = max(int(sum(p.stat().st_size for p in src_dir.rglob('*') if p.is_file()) * 0.8), 1)
        if free_bytes - est_bytes < 500 * 1024 * 1024:
            print(f"Skipping zip for {src_dir.name}: low disk space.")
            return None

        with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
            for file_path in src_dir.rglob("*"):
                if file_path.is_file():
                    zf.write(file_path, arcname=file_path.relative_to(src_dir.parent))
        return zip_path
    except OSError as exc:
        if getattr(exc, "errno", None) == errno.ENOSPC:
            print(f"Zip failed for {src_dir.name}: no space left on device.")
            return None
        raise


def sync_to_drive(paths: list[Path], cohort_label: str):
    if not SAVE_TO_DRIVE:
        return
    drive_root = Path("/content/drive/MyDrive/icl_pilot_generation_exports")
    if not drive_root.parent.exists():
        print("Drive not mounted; skipping Drive copy.")
        return

    cohort_root = drive_root / cohort_slug(cohort_label)
    cohort_root.mkdir(parents=True, exist_ok=True)
    for src in paths:
        if not src.exists():
            continue
        target = cohort_root / src.name
        if src.is_dir():
            if target.exists():
                shutil.rmtree(target)
            shutil.copytree(src, target)
        else:
            shutil.copy2(src, target)
    print(f"Saved cohort artifacts to Drive: {cohort_root}")


all_generated = []
cohort_labels = list(dict.fromkeys(run_df["cohort_label"].astype(str).tolist()))
if START_COHORT_LABEL in cohort_labels:
    cohort_labels = cohort_labels[cohort_labels.index(START_COHORT_LABEL):]
else:
    print(f"START_COHORT_LABEL not found: {START_COHORT_LABEL}. Running all cohorts.")

for cohort_label in cohort_labels:
    cohort_df = run_df[run_df["cohort_label"].astype(str) == cohort_label].copy()
    slug = cohort_slug(cohort_label)
    cohort_out_path = out_dir / f"{slug}_rotating_story_generations.csv"

    cohort_rows = []
    for row in tqdm(list(cohort_df.itertuples(index=False)), total=len(cohort_df), desc=f"Generate {cohort_label}"):
        stage1_text = generate_from_messages(
            stage1_messages(row),
            max_new_tokens=MAX_NEW_TOKENS_STAGE1,
        )
        stage2_text = generate_from_messages(
            stage2_messages(row, stage1_text),
            max_new_tokens=MAX_NEW_TOKENS_STAGE2,
        )

        cohort_rows.append({
            "cohort": row.cohort,
            "cohort_label": row.cohort_label,
            "round_id": row.round_id,
            "bundle_id": row.bundle_id,
            "pair_id": row.pair_id,
            "replicate_id": row.replicate_id,
            "story_slot_order": row.story_slot_order,
            "target_story": row.target_story,
            "story_title": row.story_title,
            "sli_child_id": row.sli_child_id,
            "td_child_id": row.td_child_id,
            "sli_age": row.sli_age,
            "sli_severity_band": row.sli_severity_band,
            "sli_profile_label": row.sli_profile_label,
            "prompt_td_story_e1": row.prompt_td_story_e1,
            "prompt_sli_story_e2": row.prompt_sli_story_e2,
            "story_units_text": row.story_units_text,
            "stage1_group": "TD",
            "stage2_group": "SLI",
            "evaluation_td_group": "TD",
            "evaluation_sli_group": "SLI",
            "stage1_td_story": stage1_text,
            "stage2_sli_story": stage2_text,
        })

        if len(cohort_rows) % SAVE_EVERY == 0:
            pd.DataFrame(cohort_rows).to_csv(cohort_out_path, index=False)

    cohort_generated_df = pd.DataFrame(cohort_rows)
    cohort_generated_df.to_csv(cohort_out_path, index=False)

    all_generated.append(cohort_generated_df)
    write_mode = "a" if aggregate_path.exists() else "w"
    include_header = write_mode == "w"
    cohort_generated_df.to_csv(aggregate_path, mode=write_mode, header=include_header, index=False)

    print(f"Saved {len(cohort_generated_df)} rows to {cohort_out_path}")
    if DOWNLOAD_EACH_COHORT_CSV:
        maybe_download(cohort_out_path)

    artifacts_to_drive = [cohort_out_path]
    local_cleanup_targets = []

    if CONVERT_TO_CHA:
        cha_root = out_dir / f"{slug}_cha"
        roster_csv = resolve_roster_csv_for_cohort(cohort_label)
        cha_info = convert_generated_story_csv_to_cha(
            input_csv=cohort_out_path,
            output_dir=cha_root,
            roster_csv=roster_csv,
            corpus_name="ICL-PILOT",
        )
        print(f"Wrote CHA files to {cha_info['output_dir']}")
        artifacts_to_drive.append(cha_root)
        artifacts_to_drive.append(Path(cha_info["manifest_csv"]))
        local_cleanup_targets.append(cha_root)

        talktag_root = out_dir / f"{slug}_talktag"
        if RUN_TALKTAG:
            run_talktag(cha_root, talktag_root)
            print(f"Wrote TalkTag outputs to {talktag_root}")
            artifacts_to_drive.append(talktag_root)
            local_cleanup_targets.append(talktag_root)

        if DOWNLOAD_CHA_AND_TALKTAG:
            cha_zip = safe_zip_dir(cha_root, out_dir / f"{slug}_cha.zip")
            if cha_zip is not None:
                maybe_download(cha_zip)
                artifacts_to_drive.append(cha_zip)
                local_cleanup_targets.append(cha_zip)

            if RUN_TALKTAG:
                talktag_zip = safe_zip_dir(talktag_root, out_dir / f"{slug}_talktag.zip")
                if talktag_zip is not None:
                    maybe_download(talktag_zip)
                    artifacts_to_drive.append(talktag_zip)
                    local_cleanup_targets.append(talktag_zip)

    sync_to_drive(artifacts_to_drive, cohort_label)

    if DELETE_LOCAL_AFTER_EXPORT:
        cleanup_paths = [cohort_out_path] + local_cleanup_targets
        for p in cleanup_paths:
            try:
                if p.is_dir() and p.exists():
                    shutil.rmtree(p)
                    print(f"Deleted local dir: {p.name}")
                elif p.is_file() and p.exists():
                    p.unlink()
                    print(f"Deleted local file: {p.name}")
            except Exception as exc:
                print(f"Could not delete {p}: {exc}")

generated_df = pd.concat(all_generated, ignore_index=True) if all_generated else pd.DataFrame()
print(f"Saved {len(generated_df)} total rows to {aggregate_path}")
generated_df[["cohort_label", "bundle_id", "target_story", "stage1_td_story", "stage2_sli_story"]].head()







## Next Step

Once the story-level rows look good, you can aggregate them by `bundle_id`, convert them into transcript-like bundle files, and then feed them into the same downstream annotation and evaluation workflow used elsewhere in the repo.

In [ ]:
# Finalize run: package outputs, download, and release GPU runtime
from pathlib import Path
import shutil
import zipfile
import errno
from datetime import datetime, timezone

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
out_csv = Path("data/generated/direct_generation/five_to_nine_year_old_rotating_story_generations.csv")
summary_json = Path("configs/generation/colab_generation_package_direct_rotating_multi_age/multi_age_rotating_generation_packages_summary.json")

package_root = Path("data/generated/direct_generation")
package_root.mkdir(parents=True, exist_ok=True)

files_to_package = [p for p in [out_csv, summary_json] if p.exists()]
if not files_to_package:
    raise FileNotFoundError("No exportable artifacts found (missing generations CSV and scaffold summary).")

zip_path = package_root / f"five_to_nine_generation_bundle_{run_stamp}.zip"


def download_paths(paths):
    try:
        from google.colab import files  # type: ignore
        for p in paths:
            files.download(str(p))
        print(f"Download started for {len(paths)} file(s).")
    except Exception as exc:
        print(f"Colab download skipped: {exc}")


def try_zip(paths, target_zip):
    # Rough safety check: leave at least 500MB free after writing estimated zip.
    total_bytes = sum(p.stat().st_size for p in paths)
    free_bytes = shutil.disk_usage(package_root).free
    estimated_zip_bytes = max(int(total_bytes * 0.8), 1)
    min_remaining = 500 * 1024 * 1024
    if free_bytes - estimated_zip_bytes < min_remaining:
        raise OSError(errno.ENOSPC, "Not enough free disk for safe zipping")

    with zipfile.ZipFile(target_zip, mode="w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for src in paths:
            arcname = src.as_posix()
            zf.write(src, arcname=arcname)


zipped = False
try:
    try_zip(files_to_package, zip_path)
    print(f"Packaged: {zip_path}")
    download_paths([zip_path])
    zipped = True
except OSError as exc:
    if getattr(exc, "errno", None) == errno.ENOSPC:
        print("Low disk space detected; skipping zip and downloading artifacts directly.")
    else:
        print(f"Zip failed ({exc}); downloading artifacts directly.")

    if zip_path.exists():
        try:
            zip_path.unlink()
        except Exception:
            pass
    download_paths(files_to_package)

# Release runtime to avoid idle GPU billing in Colab.
try:
    from google.colab import runtime  # type: ignore
    print("Unassigning runtime...")
    runtime.unassign()
except Exception as exc:
    print(f"runtime.unassign() skipped: {exc}")

